# BUV Data Processing and Validation Notebook
# 
This notebook processes and validates BUV (Baited Underwater Video) data from S3 storage.

## Setup and Dependencies

In [1]:
# Install required packages if needed
# !pip install boto3 pandas numpy tqdm

In [2]:
import logging
import os
import json
import boto3
import getpass
import pandas as pd
import numpy as np
from typing import List, Dict, Set, Union, Tuple, Iterator, Optional
from botocore.exceptions import ClientError
from tqdm import tqdm
from pathlib import Path
from dataclasses import dataclass, field

## Configure Logging

In [3]:
# Configure logging with a more detailed format
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

## AWS Credentials Management

In [4]:
@dataclass
class AWSCredentials:
    access_key_id: str
    secret_access_key: str
    
    @classmethod
    def from_user_input(cls) -> 'AWSCredentials':
        """Securely prompt user for AWS credentials."""
        access_key = getpass.getpass("Enter AWS Access Key ID: ")
        secret_key = getpass.getpass("Enter AWS Secret Access Key: ")
        return cls(access_key, secret_key)

## S3 Client Implementation

In [5]:
class S3Client:
    def __init__(self, credentials: Optional[AWSCredentials] = None):
        self.client = self._initialize_client(credentials)
    
    def _initialize_client(self, credentials: Optional[AWSCredentials]) -> boto3.client:
        """Initialize S3 client with credentials from env vars or user input."""
        if credentials is None:
            access_key = os.getenv("AWS_ACCESS_KEY_ID")
            secret_key = os.getenv("AWS_SECRET_ACCESS_KEY")
            
            if not access_key or not secret_key:
                logger.info("AWS credentials not found in environment variables.")
                credentials = AWSCredentials.from_user_input()
            else:
                credentials = AWSCredentials(access_key, secret_key)

        try:
            client = boto3.client(
                "s3",
                aws_access_key_id=credentials.access_key_id,
                aws_secret_access_key=credentials.secret_access_key,
            )
            client.list_buckets()  # Test the credentials
            logger.info("Successfully authenticated with AWS")
            return client
        except ClientError as e:
            logger.error("Failed to authenticate with AWS")
            if "InvalidAccessKeyId" in str(e) or "SignatureDoesNotMatch" in str(e):
                logger.error("Invalid credentials provided. Please try again.")
                credentials = AWSCredentials.from_user_input()
                return self._initialize_client(credentials)
            raise

    def list_objects(self, bucket: str, prefix: str = "", suffix: str = "") -> Iterator[dict]:
        """List objects in an S3 bucket with filtering."""
        paginator = self.client.get_paginator("list_objects_v2")
        
        for prefix_item in [prefix] if isinstance(prefix, str) else prefix:
            try:
                for page in paginator.paginate(Bucket=bucket, Prefix=prefix_item):
                    if "Contents" not in page:
                        continue
                    
                    for obj in page["Contents"]:
                        if obj["Key"].endswith(suffix):
                            yield obj
            except ClientError as e:
                logger.error(f"Error listing objects: {e}")
                raise

    def download_file(self, bucket: str, key: str, filename: Path) -> None:
        """Download a file from S3 with progress tracking."""
        try:
            object_size = self.client.head_object(Bucket=bucket, Key=key)["ContentLength"]
            with tqdm(total=object_size, unit='B', unit_scale=True, desc=str(filename)) as pbar:
                self.client.download_file(
                    Bucket=bucket,
                    Key=key,
                    Filename=str(filename),
                    Callback=pbar.update
                )
        except ClientError as e:
            logger.error(f"Error downloading {key}: {e}")
            raise

## CSV Data Management



In [6]:
@dataclass
class CSVCollection:
    """Container for multiple DataFrames loaded from CSV files."""
    dataframes: Dict[str, pd.DataFrame] = field(default_factory=dict)
    
    def get_df(self, key: str) -> pd.DataFrame:
        if key not in self.dataframes:
            raise KeyError(f"No DataFrame found for key: {key}")
        return self.dataframes[key]
    
    def list_dataframes(self) -> List[str]:
        """List all available DataFrame keys."""
        return list(self.dataframes.keys())

## Data Validation Functions

In [7]:
def check_missing_values(df: pd.DataFrame) -> Dict[str, int]:
    """Check for missing values in columns."""
    return {col: df[col].isna().sum() for col in df.columns}

def check_unique_values(df: pd.DataFrame, unique_columns: List[Union[str, List[str]]]) -> Dict[str, List]:
    """Check for duplicate values in columns that should be unique."""
    duplicates = {}
    
    for col_spec in unique_columns:
        if isinstance(col_spec, str):
            if col_spec in df.columns:
                dups = df[df[col_spec].duplicated()][col_spec].tolist()
                if dups:
                    duplicates[col_spec] = dups
        elif isinstance(col_spec, list) and len(col_spec) > 1:
            if all(col in df.columns for col in col_spec):
                dup_mask = df.duplicated(subset=col_spec, keep=False)
                if dup_mask.any():
                    dup_rows = df[dup_mask][col_spec]
                    dup_combinations = [tuple(row) for row in dup_rows.values.tolist()]
                    key_name = '-'.join(col_spec)
                    duplicates[key_name] = dup_combinations
    
    return duplicates

def check_reference_integrity(
    source_df: pd.DataFrame,
    reference_df: pd.DataFrame,
    source_column: str,
    reference_column: str
) -> List:
    """Check if all values in source column exist in reference column."""
    if source_column not in source_df.columns:
        raise ValueError(f"Source column '{source_column}' not found")
    if reference_column not in reference_df.columns:
        raise ValueError(f"Reference column '{reference_column}' not found")
            
    source_values = set(source_df[source_column].dropna())
    reference_values = set(reference_df[reference_column].dropna())
    
    return list(source_values - reference_values)

## CSV Validator Class


In [8]:
class CSVValidator:
    def __init__(self, s3_client: S3Client):
        self.s3_client = s3_client
        self.logger = logging.getLogger(__name__)
        self.csv_collection = None

    def convert_to_serializable(self, obj):
        """Convert values to JSON-serializable formats."""
        if isinstance(obj, (np.int64, np.int32, np.int16, np.int8)):
            return int(obj)
        elif isinstance(obj, (np.float64, np.float32, np.float16)):
            return float(obj)
        elif isinstance(obj, pd.Series):
            return obj.to_list()
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        elif isinstance(obj, dict):
            return {k: self.convert_to_serializable(v) for k, v in obj.items()}
        elif isinstance(obj, (list, tuple)):
            return [self.convert_to_serializable(item) for item in obj]
        elif pd.isna(obj):
            return None
        return obj

    def check_missing_values(self, df: pd.DataFrame) -> Dict:
        """Check for missing values in columns."""
        return {col: int(df[col].isna().sum()) for col in df.columns}

    def validate_loaded_csvs(self) -> Dict:
        """Validate all loaded CSVs for data integrity."""
        if not self.csv_collection:
            raise ValueError("No CSVs loaded. Call load_csvs_from_prefix first.")

        validation_results = {
            'missing_values': {},
            'unique_values': {},
            'reference_integrity': {},
            'missing_columns': {}
        }

        # Define expected columns for each CSV
        expected_columns = {
            'BUV Deployment': ['SurveyID', 'DropID'],
            'BUV Survey Metadata': ['SurveyID', 'SiteID'],
            'BUV Survey Sites': ['SiteID']
        }

        # Define validation rules for each CSV
        validation_rules = {
            'BUV Deployment': {
                'unique_columns': [['SurveyID', 'DropID']],
                'reference_mappings': {
                    'SurveyID': {'reference': 'BUV Survey Metadata', 'column': 'SurveyID'}
                }
            },
            'BUV Survey Metadata': {
                'unique_columns': ['SurveyID'],
                'reference_mappings': {
                    'SiteID': {'reference': 'BUV Survey Sites', 'column': 'SiteID'}
                }
            },
            'BUV Survey Sites': {
                'unique_columns': ['SiteID']
            }
        }

        # Process each DataFrame
        for csv_name, df in self.csv_collection.dataframes.items():
            # Print DataFrame info for debugging
            self.logger.info(f"\nProcessing {csv_name}")
            self.logger.info(f"Columns: {df.columns.tolist()}")
            self.logger.info(f"Shape: {df.shape}")

            # Check for missing columns
            if csv_name in expected_columns:
                missing_cols = [col for col in expected_columns[csv_name] if col not in df.columns]
                if missing_cols:
                    validation_results['missing_columns'][csv_name] = missing_cols
                    self.logger.warning(f"Missing columns in {csv_name}: {missing_cols}")
                    continue

            # Check missing values
            validation_results['missing_values'][csv_name] = self.check_missing_values(df)

            # Check unique constraints
            if csv_name in validation_rules and 'unique_columns' in validation_rules[csv_name]:
                try:
                    duplicates = {}
                    for cols in validation_rules[csv_name]['unique_columns']:
                        if isinstance(cols, str):
                            cols = [cols]
                        if all(col in df.columns for col in cols):
                            dups = df[df.duplicated(subset=cols, keep=False)]
                            if not dups.empty:
                                key = '_'.join(cols)
                                duplicates[key] = dups[cols].values.tolist()
                    if duplicates:
                        validation_results['unique_values'][csv_name] = duplicates
                except Exception as e:
                    self.logger.error(f"Error checking unique values for {csv_name}: {str(e)}")

            # Check reference integrity
            if csv_name in validation_rules and 'reference_mappings' in validation_rules[csv_name]:
                validation_results['reference_integrity'][csv_name] = {}
                for source_col, ref_info in validation_rules[csv_name]['reference_mappings'].items():
                    try:
                        if source_col not in df.columns:
                            continue
                        if ref_info['reference'] not in self.csv_collection.dataframes:
                            continue
                        ref_df = self.csv_collection.get_df(ref_info['reference'])
                        if ref_info['column'] not in ref_df.columns:
                            continue
                        
                        invalid_refs = check_reference_integrity(
                            df,
                            ref_df,
                            source_col,
                            ref_info['column']
                        )
                        validation_results['reference_integrity'][csv_name][source_col] = [
                            str(val) for val in invalid_refs
                        ]
                    except Exception as e:
                        self.logger.error(f"Error checking reference integrity for {csv_name}.{source_col}: {str(e)}")

        # Convert all values to JSON-serializable format
        return self.convert_to_serializable(validation_results)

    def load_csvs_from_prefix(self) -> CSVCollection:
        """Load all CSV files from a given S3 prefix."""
        try:
            bucket = os.getenv("S3_BUCKET")
            if not bucket:
                logger.info("AWS Bucket not found in environment variables.")
                bucket = getpass.getpass("Enter AWS Bucket name:")
            
            prefix = os.getenv("S3_SHAREPOINT_CSVS")
            if not prefix:
                logger.info("AWS Key with csv files not found in environment variables.")
                prefix = getpass.getpass("AWS Key of with csv files:")
            
            csv_objects = self.s3_client.list_objects(bucket, prefix=prefix, suffix=".csv")
            self.csv_collection = CSVCollection()
            
            for obj in csv_objects:
                key = obj["Key"]
                filename = Path(key).name
                temp_path = Path(f"temp_{filename}")
                
                try:
                    self.s3_client.download_file(bucket, key, temp_path)
                    df = pd.read_csv(temp_path)
                    df_key = filename.replace('.csv', '')
                    self.csv_collection.dataframes[df_key] = df
                    self.logger.info(f"Successfully loaded {filename}")
                finally:
                    if temp_path.exists():
                        temp_path.unlink()
            
            return self.csv_collection
            
        except Exception as e:
            self.logger.error(f"Error loading CSVs from prefix {prefix}: {e}")
            raise  

    def print_dataframe_info(self):
        """Print information about loaded DataFrames."""
        if not self.csv_collection:
            print("No DataFrames loaded")
            return
            
        for name, df in self.csv_collection.dataframes.items():
            print(f"\nDataFrame: {name}")
            print("Columns:", df.columns.tolist())
            print(f"Shape: {df.shape}")
            print("First few rows:")
            print(df.head())

## Data Processing Functions

In [9]:
def check_and_export_differences(df: pd.DataFrame, output_path: str = 'column_differences.csv') -> tuple[pd.DataFrame, bool]:
    """Identifies and exports duplicate columns with different values."""
    df_cleaned = df.copy()
    has_differences = False
    diff_rows = []
    
    base_columns = {col[:-2] for col in df.columns if col.endswith('_x') or col.endswith('_y')}
    
    for base_col in base_columns:
        x_col = f"{base_col}_x"
        y_col = f"{base_col}_y"
        
        if x_col in df.columns and y_col in df.columns:
            try:
                x_series = pd.to_numeric(df[x_col], errors='ignore')
                y_series = pd.to_numeric(df[y_col], errors='ignore')
                are_equal = x_series.equals(y_series)
                
                if not are_equal:
                    x_str = df[x_col].astype(str)
                    y_str = df[y_col].astype(str)
                    are_equal = x_str.equals(y_str)
            except Exception as e:
                logger.warning(f"Error comparing {x_col} and {y_col}: {str(e)}")
                are_equal = False
            
            if are_equal:
                df_cleaned[base_col] = df_cleaned[x_col]
                df_cleaned = df_cleaned.drop([x_col, y_col], axis=1)
            else:
                df_cleaned[base_col] = df_cleaned[x_col]
                df_cleaned = df_cleaned.drop([x_col, y_col], axis=1)
    
    return df_cleaned, has_differences

## Main Processing Pipeline

In [10]:
def merge_buv_metadata(csv_collection: CSVCollection) -> pd.DataFrame:
    """Merge all BUV metadata dataframes."""
    if not csv_collection:
        raise ValueError("No CSVs loaded. Call load_csvs_from_prefix first.")

    has_any_differences = False
    
    # First merge: Deployment and Survey Metadata
    df = csv_collection.get_df("BUV Deployment").merge(
        csv_collection.get_df("BUV Survey Metadata"), 
        on="SurveyID", 
        how="left"
    )
    df, has_diff = check_and_export_differences(df, 'differences_merge_Deployment_Survey.csv')
    has_any_differences |= has_diff
    
    # Second merge: Add Survey Sites
    df = df.merge(
        csv_collection.get_df("BUV Survey Sites"), 
        on="SiteID", 
        how="left"
    )
    df, has_diff = check_and_export_differences(df, 'differences_merge_Deployment_Survey_Sites.csv')
    has_any_differences |= has_diff
    
    # Third merge: Add Marine Reserves
    df = df.merge(
        csv_collection.get_df("Marine Reserves"), 
        left_on="LinkToMarineReserve", 
        right_on="Title", 
        how="left"
    )
    df, has_diff = check_and_export_differences(df, 'differences_merge_Deployment_Survey_Sites_MReserves.csv')
    has_any_differences |= has_diff
    
    if not has_any_differences:
        logger.info("No differences found in any merge operations")
    
    return df

## Run the Pipeline


In [11]:
# Initialize S3 client and validator
s3_client = S3Client()
validator = CSVValidator(s3_client)

# Load all CSVs from a common prefix
csv_collection = validator.load_csvs_from_prefix()

# Print information about loaded DataFrames
print("\nLoaded DataFrame Information:")
validator.print_dataframe_info()

# Run validation
validation_results = validator.validate_loaded_csvs()

# Print results
print("\nValidation Results:")
print(json.dumps(validation_results, indent=2))

# Create a DataFrame with all buv metadata
buv_metadata = merge_buv_metadata(csv_collection)

2025-02-14 15:18:31,197 - __main__ - INFO - Successfully authenticated with AWS
temp_BUV Deployment.csv: 100%|██████████| 205k/205k [00:00<00:00, 942kB/s]
2025-02-14 15:18:32,221 - __main__ - INFO - Successfully loaded BUV Deployment.csv
temp_BUV Metadata Definitions.csv: 100%|██████████| 49.9k/49.9k [00:00<00:00, 466kB/s]
2025-02-14 15:18:32,403 - __main__ - INFO - Successfully loaded BUV Metadata Definitions.csv
temp_BUV Species.csv: 100%|██████████| 14.6k/14.6k [00:00<00:00, 123kB/s]
2025-02-14 15:18:32,596 - __main__ - INFO - Successfully loaded BUV Species.csv
temp_BUV Survey Metadata.csv: 100%|██████████| 17.3k/17.3k [00:00<00:00, 174kB/s]
2025-02-14 15:18:32,762 - __main__ - INFO - Successfully loaded BUV Survey Metadata.csv
temp_BUV Survey Sites.csv: 100%|██████████| 167k/167k [00:00<00:00, 1.30MB/s]
2025-02-14 15:18:32,970 - __main__ - INFO - Successfully loaded BUV Survey Sites.csv
temp_Marine Reserves.csv: 100%|██████████| 13.1k/13.1k [00:00<00:00, 118kB/s]
2025-02-14 15:18:


Loaded DataFrame Information:

DataFrame: BUV Deployment
Columns: ['DropID', 'SurveyID', 'SiteID', 'Latitude', 'Longitude', 'EventDate', 'Created By', 'TideLevel', 'Weather', 'UnderwaterVisibility', 'ReplicateWithinSite', 'EventTimeStart', 'EventTimeEnd', 'DepthDeployment', 'DepthStrata', 'NZMHCS_Abiotic', 'NZMHCS_Biotic', 'NotesDeployment', 'RecordedBy', 'IsBadDeployment', 'fps', 'duration', 'fileName', 'LinkToVideoFile', 'SamplingStart', 'SamplingEnd', 'ID']
Shape: (712, 27)
First few rows:
                  DropID          SurveyID   SiteID  Latitude  Longitude  \
0  SLI_20240124_BUV_0001  SLI_20240124_BUV  SLI_085 -39.08958  173.96091   
1  SLI_20240124_BUV_0002  SLI_20240124_BUV  SLI_089 -36.89340  174.70990   
2  SLI_20240124_BUV_0003  SLI_20240124_BUV  SLI_088 -36.89340  174.70990   
3  SLI_20240124_BUV_0004  SLI_20240124_BUV  SLI_087 -36.89340  174.70990   
4  SLI_20240124_BUV_0005  SLI_20240124_BUV  SLI_044 -36.89340  174.70990   

    EventDate    Created By      TideLevel  

## Save Results


In [12]:
# Save the processed data
output_path = "processed_buv_data.csv"
buv_metadata.to_csv(output_path, index=False)
print(f"Processed data saved to {output_path}")

Processed data saved to processed_buv_data.csv
